In [4]:
%matplotlib inline

import os
import random
import numpy as np
import pandas as pd
import yfinance as yf
import gymnasium as gym
from gymnasium import spaces
import optuna
from datetime import datetime
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback

# from finrl.agents import DRLAgent
# from finrl.config import config
from finrl.meta.env_stock_trading.env_stocktrading_np import StockTradingEnv


import random

# import matplotlib
import matplotlib.pyplot as plt

import MetaTrader5 as mt5

import time

In [5]:
MT5_LOGIN = int(os.environ.get('MT5_LOGIN'))
MT5_PASSWORD = os.environ.get('MT5_PASSWORD')
MT5_SERVER = os.environ.get('MT5_SERVER')
SYMBOL = "EURUSD"
TIMEFRAME = mt5.TIMEFRAME_D1
MODEL_PATH = "models/best_ppo_forex.zip"
RESUME_TIMESTEPS = 100_000
START = pd.Timestamp("2024-01-01")
END = pd.Timestamp("2025-10-01")
PARQUET_PATH = f"data/{SYMBOL.lower()}_{TIMEFRAME}.parquet"

In [6]:
# --- Fonction de connexion ---
def connect_mt5(max_retries=3, wait_time=2):
    """Connexion robuste à MetaTrader 5"""
    print("🔹 Fermeture d'éventuelles connexions précédentes...")
    mt5.shutdown()
    time.sleep(1)

    for attempt in range(1, max_retries + 1):
        print(f"\n🔄 Tentative {attempt}/{max_retries} d'initialisation de MetaTrader 5...")
        if not mt5.initialize():
            err = mt5.last_error()
            print(f"❌ Échec de l'initialisation : {err}")
            time.sleep(wait_time)
            continue

        mt5_version = mt5.version()
        print(f"✅ MetaTrader 5 initialisé : version {mt5_version}")

        if MT5_LOGIN:
            if not mt5.login(login=MT5_LOGIN, password=MT5_PASSWORD, server=MT5_SERVER):
                err = mt5.last_error()
                print(f"❌ Connexion échouée : {err}")
                mt5.shutdown()
                time.sleep(wait_time)
                continue
            print("✅ Connexion réussie à MetaTrader 5.")
        return True

    return False


# --- Chargement ou téléchargement des données ---
if os.path.exists(PARQUET_PATH):
    print(f"📁 Chargement des données existantes depuis {PARQUET_PATH}")
    df_mt5 = pd.read_parquet(PARQUET_PATH)

else:
    print("🌐 Téléchargement des données depuis MetaTrader 5...")
    if connect_mt5():
        rates = mt5.copy_rates_range(SYMBOL, TIMEFRAME, START.to_pydatetime(), END.to_pydatetime())
        if rates is None:
            raise Exception("❌ Aucune donnée reçue depuis MT5.")
        
        df_mt5 = pd.DataFrame(rates)
        if df_mt5.empty:
            raise Exception("❌ Données vides — vérifie la période et le symbole.")

        # Mise en forme
        df_mt5["time"] = pd.to_datetime(df_mt5["time"], unit="s")
        df_mt5.columns = df_mt5.columns.str.lower()  # ✅ Colonnes en minuscules
        df_mt5.reset_index(drop=True, inplace=True)

        print(f"✅ {len(df_mt5)} barres téléchargées.")
        os.makedirs(os.path.dirname(PARQUET_PATH), exist_ok=True)
        df_mt5.to_parquet(PARQUET_PATH, index=False)
        print(f"💾 Données sauvegardées dans {PARQUET_PATH}")
        mt5.shutdown()
    else:
        raise Exception("❌ Impossible de se connecter à MT5.")


# --- Aperçu ---
print(df_mt5.head())

🌐 Téléchargement des données depuis MetaTrader 5...
🔹 Fermeture d'éventuelles connexions précédentes...

🔄 Tentative 1/3 d'initialisation de MetaTrader 5...
✅ MetaTrader 5 initialisé : version (500, 5370, '17 Oct 2025')
✅ Connexion réussie à MetaTrader 5.
✅ 455 barres téléchargées.
💾 Données sauvegardées dans data/eurusd_16408.parquet
        time     open     high      low    close  tick_volume  spread  \
0 2024-01-02  1.10437  1.10451  1.09374  1.09407       205171       0   
1 2024-01-03  1.09425  1.09655  1.08925  1.09212       234801       0   
2 2024-01-04  1.09205  1.09723  1.09056  1.09435       195260       0   
3 2024-01-05  1.09443  1.09988  1.08769  1.09408       263575       0   
4 2024-01-08  1.09331  1.09789  1.09225  1.09496       183411       0   

   real_volume  
0            0  
1            0  
2            0  
3            0  
4            0  


In [7]:
# --- Préparer l'environnement FinRL pour le fine-tuning ---
config_ft = {
    "price_array": df_mt5["close"].values.reshape(-1, 1),
    "tech_array": df_mt5[["close"]].values,  # ou tes features
    "turbulence_array": pd.Series(0, index=df_mt5.index).values,  # si tu n'as pas de turbulence
    "if_train": True,
}

In [8]:
class HallucinationCallback(BaseCallback):
    def __init__(self, env, verbose=0):
        super().__init__(verbose)
        self.env = env
        self.history = []  # Pour stocker les historiques

    def _on_step(self) -> bool:
        # Récupérer les informations du portefeuille
        balance = self.env.total_asset
        position = self.env.stocks  # ou l'information que tu veux suivre, en fonction de l'environnement

        # Historique
        self.history.append((balance, position))

        # Détection d'hallucination
        if len(self.history) > 1:
            prev_balance, _ = self.history[-2]
            if balance < 0.5 * prev_balance:  # Condition d'hallucination basique
                print(f"⚠️ Alerte hallucination : balance {balance:.2f}, position {position}")
                return False  # Stop training si hallucination détectée

        return True  # Continuer l'entraînement

In [9]:
def fine_tune_model(model, env, total_timesteps=100_000):
    halluc_cb = HallucinationCallback(env)
    
    # Entraînement avec le callback
    model.learn(total_timesteps=total_timesteps, callback=halluc_cb)
    
    return model, halluc_cb

In [11]:
env_ft = DummyVecEnv([lambda: StockTradingEnv(config=config_ft)])

model_path = "models/ppo_forex_final.zip"
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Le modèle n'existe pas : {model_path}")

model = PPO.load(model_path, env=env_ft, verbose=1)

# Lancer le fine-tuning avec détection d'hallucination
model, halluc_cb = fine_tune_model(model, env_ft, total_timesteps=100_000)

ValueError: Observation spaces do not match: Box(-3000.0, 3000.0, (8,), float32) != Box(-3000.0, 3000.0, (7,), float32)

In [ ]:
def plot_performance(actions_list, total_assets, filename="performance_plot.png"):
    actions_list = np.array(actions_list)
    buy_points = np.where(actions_list > 0)[0]
    sell_points = np.where(actions_list < 0)[0]

    # --- Tracé du portefeuille ---
    plt.figure(figsize=(12, 6))
    plt.plot(total_assets, label="Valeur du portefeuille", color="blue")

    # Points d'achat et vente
    plt.scatter(buy_points, np.array(total_assets)[buy_points], color="green", label="Achat", marker="^", alpha=0.8)
    plt.scatter(sell_points, np.array(total_assets)[sell_points], color="red", label="Vente", marker="v", alpha=0.8)

    plt.title("📈 Performance du modèle FinRL sur EUR/USD")
    plt.xlabel("Étapes (jours)")
    plt.ylabel("Valeur du portefeuille (€)")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Sauvegarder le graphique
    plt.savefig(filename)
    print(f"📊 Graphique sauvegardé sous {filename}")

In [ ]:
actions_list = []
total_assets = []
obs = env_ft.reset()

for _ in range(env_ft.max_step):
    action, _ = model.predict(obs, deterministic=True)
    actions_list.append(action)
    obs, reward, done, info = env_ft.step(action)
    total_assets.append(env_ft.total_asset)

    if done:
        break

plot_performance(actions_list, total_assets)

In [ ]:
ft_model_dir = "models/fine_tuned"
os.makedirs(ft_model_dir, exist_ok=True)
ft_model_path = os.path.join(ft_model_dir, "ppo_forex_finetuned.zip")
model.save(ft_model_path)

print(f"💾 Modèle fine-tuné sauvegardé : {ft_model_path}")